## **Rede Perceptron — Classificação de Clientes**

**Atividade 2:** Classificar clientes como *bom* ou *mau* pagador  
usando Renda e Dívida como atributos de entrada.

- **70%** dos dados → Treinamento  
- **30%** dos dados → Teste

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

## 1. Carregamento e Análise dos Dados

In [ ]:
df = pd.read_csv('dados.csv')
print('Shape:', df.shape)
print()
print(df)
print()
print('Distribuição das classes:')
print(df['Classe'].value_counts())

## 2. Seleção de Atributos

**Atributos escolhidos:** `Renda` e `Divida`

- **Renda**: quanto maior a renda, maior a chance de ser bom pagador
- **Dívida**: quanto maior a dívida em relação à renda, maior o risco
- **Cliente**: apenas um identificador, sem valor preditivo → descartado

**Classe (alvo):** `bom` → 1, `mau` → 0

In [ ]:
# Selecionar atributos relevantes
X = df[['Renda', 'Divida']].values
y = df['Classe'].map({'bom': 1, 'mau': 0}).values

print('Entradas (X):'); print(X)
print('Saídas  (y):', y)

## 3. Pré-processamento — Normalização Min-Max

Renda varia de ~450 a 2800 e Dívida de ~70 a 800.  
Sem normalização, o peso de Renda dominaria o treinamento.  
A normalização traz tudo para o intervalo **[0, 1]**:

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

In [ ]:
x_min = X.min(axis=0)
x_max = X.max(axis=0)

X_norm = (X - x_min) / (x_max - x_min)

print('Min por coluna:', x_min)
print('Max por coluna:', x_max)
print()
print('Dados normalizados:')
print(np.round(X_norm, 3))

## 4. Divisão Treino / Teste (70% / 30%)

In [ ]:
random.seed(42)
n = len(X_norm)
indices = list(range(n))
random.shuffle(indices)

n_treino = int(n * 0.7)   # 70% → 14 amostras
n_teste  = n - n_treino   # 30% →  7 amostras

idx_treino = indices[:n_treino]
idx_teste  = indices[n_treino:]

# Montar tuplas [x1, x2, d] para o algoritmo de treinamento
dados_treino = [[X_norm[i][0], X_norm[i][1], y[i]] for i in idx_treino]
dados_teste  = [[X_norm[i][0], X_norm[i][1], y[i]] for i in idx_teste]

print(f'Total de amostras : {n}')
print(f'Amostras treino   : {n_treino}')
print(f'Amostras teste    : {n_teste}')
print()
print('Dados de treino (x1_norm, x2_norm, classe):')
for d in dados_treino:
    print(f'  Renda={d[0]:.3f}  Divida={d[1]:.3f}  Classe={d[2]}')

## 5. Funções do Perceptron (mesma topologia da atividade anterior)

In [ ]:
def funcaoLimiar(v):
    return 1 if v >= 0 else 0

def propagacao(tupla, pesos):
    v = pesos[0] + pesos[1] * tupla[0] + pesos[2] * tupla[1]
    return v

def algoritmoTreinamento(dados, eta=0.1, max_epocas=200):
    pesos = [0.0, 0.0, 0.0]
    print('Pesos iniciais:', pesos)
    erros_por_epoca = []

    for epoca in range(1, max_epocas + 1):
        erroGeral = 0
        for tupla in dados:
            v = propagacao(tupla, pesos)
            y = funcaoLimiar(v)
            erro = tupla[2] - y
            if erro != 0:
                pesos[0] += eta * erro
                pesos[1] += eta * erro * tupla[0]
                pesos[2] += eta * erro * tupla[1]
            erroGeral += abs(erro)

        erros_por_epoca.append(erroGeral)
        if erroGeral == 0:
            print(f'Convergiu na época {epoca}!')
            break
    else:
        print(f'Atingiu {max_epocas} épocas sem convergir total (erro final={erroGeral})')

    print('Pesos finais:', [round(p, 4) for p in pesos])
    return pesos, erros_por_epoca

## 6. Treinamento

In [ ]:
pesos, historico_erros = algoritmoTreinamento(dados_treino, eta=0.1, max_epocas=200)

# Gráfico de convergência
plt.figure(figsize=(8, 4))
plt.plot(historico_erros, marker='o', color='steelblue')
plt.xlabel('Época')
plt.ylabel('Erro total')
plt.title('Convergência do Perceptron — Clientes')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 7. Teste e Métricas de Desempenho

In [ ]:
print('=' * 55)
print(f'  {"Cliente":>8}  {"Renda":>8}  {"Divida":>8}  {"Real":>5}  {"Pred":>5}  {"OK?":>4}')
print('=' * 55)

acertos = 0
for i, tupla in zip(idx_teste, dados_teste):
    v = propagacao(tupla, pesos)
    pred = funcaoLimiar(v)
    real = int(tupla[2])
    ok = '✅' if pred == real else '❌'
    if pred == real:
        acertos += 1
    cliente = df.iloc[i]['Cliente']
    renda   = df.iloc[i]['Renda']
    divida  = df.iloc[i]['Divida']
    classe_real = 'bom' if real == 1 else 'mau'
    classe_pred = 'bom' if pred == 1 else 'mau'
    print(f'  {cliente:>8}  {renda:>8}  {divida:>8}  {classe_real:>5}  {classe_pred:>5}  {ok}')

print('=' * 55)
acuracia = acertos / n_teste * 100
print(f'Acertos : {acertos}/{n_teste}')
print(f'Acurácia: {acuracia:.1f}%')

## 8. Visualização — Fronteira de Decisão

In [ ]:
w0, w1, w2 = pesos

fig, ax = plt.subplots(figsize=(8, 6))

# Plotar todos os pontos (treino=círculo, teste=estrela)
for i in range(n):
    cor    = 'green' if y[i] == 1 else 'red'
    marker = 'o' if i in idx_treino else '*'
    size   = 80   if i in idx_treino else 200
    ax.scatter(X_norm[i][0], X_norm[i][1], c=cor, marker=marker,
               s=size, edgecolors='black', linewidths=0.5)
    ax.annotate(str(df.iloc[i]['Cliente']),
                (X_norm[i][0], X_norm[i][1]),
                textcoords='offset points', xytext=(5, 5), fontsize=7)

# Reta de separação: w0 + w1*x1 + w2*x2 = 0  →  x2 = (-w0 - w1*x1) / w2
x_line = np.linspace(-0.05, 1.05, 200)
if w2 != 0:
    y_line = (-w0 - w1 * x_line) / w2
    ax.plot(x_line, y_line, 'k--', linewidth=2, label='Fronteira de decisão')
else:
    ax.axvline(x=-w0/w1, color='black', linestyle='--', label='Fronteira de decisão')

# Legenda manual
from matplotlib.lines import Line2D
legenda = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='green', markersize=10, label='Treino — bom'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='red',   markersize=10, label='Treino — mau'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='green', markersize=14, label='Teste  — bom'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='red',   markersize=14, label='Teste  — mau'),
    Line2D([0],[0], color='black', linestyle='--', label='Fronteira de decisão'),
]
ax.legend(handles=legenda, loc='upper right', fontsize=9)

ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.set_xlabel('Renda (normalizada)')
ax.set_ylabel('Dívida (normalizada)')
ax.set_title('Perceptron — Classificação de Clientes\n(⭘ treino | ★ teste)')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()